In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")
adata

In [ ]:
adata.obs['state'].value_counts()

In [ ]:
sc.pl.umap(
    adata,
    color='celltype',
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    adata,
    color='celltype',
    frameon=False,
    size=10,
    show=False,       
    return_fig=True   
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Cell_total_celltypist.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
sc.pl.umap(
    adata,
    color='state',
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    adata,
    color=['CD3D','CD3E','CD3G'],
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    adata,
    color=['CD4','CD8B'],
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    adata,
    color='cellannot_detail_JHB',
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    adata,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='SLC7A5 expression',
    size=10
)

In [ ]:
sc.pl.violin(adata,"SLC7A5" ,groupby = 'state')

In [ ]:
sc.pl.violin(adata,"CD8B" ,groupby = 'state')

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scraw.obs['celltype'] = adata.obs['celltype']
adata_scraw.obs['cellannot_detail_JHB'] = adata.obs['cellannot_detail_JHB']

In [ ]:
adata_T1 = adata_scraw[adata_scraw.obs['cellannot_detail_JHB'].isin(['T cells'])].copy()
adata_T1

In [ ]:
adata_T = adata_scraw[adata_scraw.obs['celltype'].isin(['T cells'])].copy()
adata_T

In [ ]:
sc.pp.normalize_total(adata_T1, target_sum=1e4)
sc.pp.log1p(adata_T1)
adata_T1.raw = adata_T1
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_T1, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_T1, max_value=10)
# Run PCA
sc.tl.pca(adata_T1, svd_solver='arpack')
print(adata_T1.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_T1, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_T1.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_T1, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_T1, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_T1)
sc.pl.umap(adata_T1, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_T1.obsm['X_pca'],adata_T1.obs, 'state')

# Harmony 결과를 저장
adata_T1.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_T1, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_T1)
sc.pl.umap(adata_T1, color=['state'])

In [ ]:
sc.pl.umap(
    adata_T1,
    color='state',
    frameon=False,
    title='sc_T_Harmony',
    size=10,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    return_fig=True    # <- 중요: 그림 객체 반환
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Cell_T_Harmony.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
sc.pl.umap(adata_T1, color=['state'])

In [ ]:
sc.pl.umap(adata_T1, color=['CD4'])

In [ ]:
sc.tl.leiden(adata_T1, resolution=0.27, key_added="leiden")
sc.pl.umap(adata_T1, color=['leiden'])

In [ ]:
adata_T1.obs['state'].value_counts()